# STAC AI Workshop · NYC · 21 October 2026

## Section 4 · Actual LoRA adaptation: inspect, compare, qualify

**Participant workbook · Matthew Dixon, PhD, FRM**

STAC AI Workshop NYC · 21 October 2026

**Learning objectives**
1. Inspect actual SFT records and a saved LoRA configuration.
2. Compare base and adapter using identical prompts and evidence.
3. Recompute accuracy, per-class recall and balanced accuracy.
4. Separate evidence eligibility from forecast correctness.
5. State leakage and generalization limits before qualification.

Default mode replays **saved measured Qwen2.5-0.5B outputs**. No API, GPU or training runs. A/B/C are class-token outputs, not prose or calibrated probabilities. No measured separate RAG arm exists. Both compared models receive the same evidence. The instructor training notebook is separate.

In [ ]:
from pathlib import Path
import sys, json
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "adaptation_lab.py").exists()), None)
if root is None: raise RuntimeError("Open this notebook inside the extracted workshop package.")
sys.path.insert(0, str(root))
from adaptation_lab import load, verify, metrics, local_score
import adaptation_display as view
data = load()
m = data["metadata"]
view.status(verify(data), m["execution"])


## 1 · Inspect what was trained
SFT supplies the objective; LoRA supplies the parameterization. Query/value adapters use rank 8, alpha 16. The original 494,032,768 parameters remain frozen. The adapter adds 540,672 trainable parameters, verified from saved tensor shapes.

In [ ]:
view.model(data)


## 2 · Inspect loss and checkpoint selection
These are recorded losses, not a new run. Epoch 2 was selected using earlier validation loss. Lower training loss does not establish improved held-out forecasting. Three non-finite gradient-norm log entries are retained as an audit limitation; saved evaluated logits are finite.

In [ ]:
view.losses(data)


## 3 · Choose a fixed held-out case
Change CASE_INDEX to inspect another recorded case. Predict negative, neutral or positive **before** revealing the observed outcome. The outcome is an evaluation label, not a fact the model knew.

In [ ]:
CASE_INDEX = 0  # Choose 0–15; rerun this cell and the reveal cell below.
case = data["cases"][CASE_INDEX]
assert case["base"]["prompt_sha256"] == case["adapted"]["prompt_sha256"]
view.case_view(case)


## 4 · Reveal gold and inspect independent checks
Eligibility means the supplied records pass specified metadata/calculation checks. It does not prove that news is relevant or the forecast correct. For example, examine whether a news headline about other oil firms actually supports the target issuer.

In [ ]:
view.reveal(case)


## 5 · Recompute aggregate results
All 16 cases remain in the evaluation; do not choose a winning subset. The test has only one neutral outcome. Correctly predicting it moves balanced accuracy substantially. Compare raw accuracy and class counts too.

In [ ]:
assert metrics(data,"base")["accuracy"] == 6/16
assert metrics(data,"adapted")["accuracy"] == 3/16
view.results(data, metrics)


## 6 · Check temporal separation and qualification
Training outcomes precede April 1; validation outcomes precede June 5; July is the fixed evaluation block. The same four issuers appear across splits. Sources were downloaded later: these are not certified contemporaneous snapshots. A displayed historical publication date cannot prove historical content. No positive alpha was established.

In [ ]:
view.protocol(data)


## Optional · Fresh inference (explicitly off)
Requires approved local base and adapter files plus PyTorch, Transformers and PEFT. The function checks artifact hashes and rendered prompt hash before inference. It never downloads weights. CPU inference can be slow; the saved path is the workshop default. Changing input requires a separate fresh run and a new evaluation design; saved outputs do not update.

In [ ]:
RUN_FRESH=False
if RUN_FRESH:
 BASE_DIR=Path("/models/base") if Path("/models/base").exists() else root/"instructor-sandbox/models/base"
 ADAPTER_DIR=Path("/app/artifacts/alpha_v2/adapter") if Path("/app/artifacts/alpha_v2/adapter").exists() else root/"instructor-sandbox/artifacts/alpha_v2/adapter"
 view.fresh(local_score(case,BASE_DIR,ADAPTER_DIR,m["freeze"]["base_sha256"],m["freeze"]["adapter_sha256"]))
else:view.fresh()


## Learning check and exit ticket
- What was trained: a small adapter or a smaller replacement base model?
- Why is identical evidence important when comparing base and adapter?
- Why is higher balanced accuracy here insufficient evidence of a better model?
- Which evidence checks passed without proving relevance or forecasting skill?

**Debrief:** 540,672 adapter parameters were trained. The adapter changed behaviour, but July raw accuracy fell from 6/16 to 3/16. Balanced accuracy is sensitive to the single neutral case. Save a one-paragraph qualify/do-not-qualify decision with the specific evidence supporting it.